# 第1章：LLM基础
什么是大语言模型LLM：语言模型，是一种设计用于理解，生成文本的模型，通常采用注意力机制的神经网络组成。大是指类似GPT3有着数以千亿级别的参数量（1750亿）
1. 如何理解输入文本prompt的？
- 输入文本进行id化，id代表token在词表中的位置
- 通过嵌入矩阵进行词嵌入，得到词嵌入向量。
- 嵌入矩阵负责 Token embedding，shape: vocab_size * embedding_dim，它的参数随不同模型采用语料的不同，词表大小不同，嵌入维度也不同。但不论什么模型一定有这一部分。当然 纯图像模型（ViT） 语音模型 没有这个。
- 位置编码 Positional embedding, shape: max_seq_len * embedding_dim 负责位置信息，是另一套独立参数。 这里的max_seq_len就是上下文长度。
- RoPE没有位置编码，RoPE没有可学习的位置嵌入矩阵，对 query 和 key 向量按照位置索引做旋转  
2. 如何生成文本的，如何针对不同的要求生成不同的文本？
- 后面会讲
3. 一次可输入的词的个数有无要求？
- 上下文长度 = 输入 Prompt + 生成输出 的总 Token 数
- Transformer 的自注意力机制是对「整个历史序列」做统一计算的：新生成的token是可以看到之前所有位置的token的。
- 从 KV 缓存视角看：生成阶段的 KV 缓存存储了所有历史 token 的键值向量，缓存大小和总序列长度成正比，总长度不能超过窗口设计的最大值，否则会超出显存和结构限制。  
- 滑动窗口生成：看似能无限生成，实则是「用遗忘换持续」  序列截断式滑动：当总 token 数快要触达窗口上限时，丢弃序列最前面的一部分旧 token，保留尾部的最新 token，腾出空间继续生成。   必然丢失前文记忆，逻辑断裂。Transformer 依赖位置编码来识别 token 的顺序关系  
- 行业里的 “超长文本生成” 方案（比如长文档写作工具），本质都是滑动窗口 + 关键信息锚定—— 手动把核心设定、大纲等关键信息固定在窗口里不被滑动掉，普通正文滑动丢弃，以此在有限窗口里尽量维持长文本的连贯性，但依然不是真正的无限上下文。  

# 第2章：处理文本数据

本笔记本使用的软件包：

In [20]:
from importlib.metadata import version

import tiktoken
import torch
import re

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

torch version: 2.14.1
tiktoken version: 0.14.0


## 2.2 文本分词

- 所需基础知识

In [13]:
tensor1 = torch.randn(3,2)
# 使用tensor.max(dim=1)来获取每个样本的最大值，但返回的是一个元组，包含最大值和最大值的索引
# 我们只需要最大值，所以可以使用tensor.max(dim=1)[0]来获取最大值
print(tensor1.max(dim=1))

# 它会降维，返回的是一个一维张量，包含每个样本的最大值，原始3个样本，所以该张量的形状为(3,)
print(tensor1.max(dim=1)[0])


# 二维张量转列表
list1 = tensor1.tolist()
# 返回的是一个嵌套列表，共3个元素，每个元素是一个列表，包含2个元素，分别是每个样本的2个特征
print(list1)

torch.return_types.max(
values=tensor([1.6271, 0.2543, 0.7907]),
indices=tensor([1, 0, 0]))
tensor([1.6271, 0.2543, 0.7907])
[[-1.481791377067566, 1.6271270513534546], [0.2543310821056366, -0.11790447682142258], [0.7907112836837769, -0.4818825423717499]]


In [18]:
vocab_test = {'hello':1,"world":2}
for s,i in vocab_test.items():
    # 打印每个键值对
    print(s,i)

# 直接使用list()函数将字典转换为列表
list1 = list(vocab_test.items())
print(list1)

# 如果不使用items方法提取键值对，默认提取键
list2 = list(vocab_test)
print(list2)


hello 1
world 2
[('hello', 1), ('world', 2)]
['hello', 'world']


In [19]:
list3 = [" hello "," ","\n","\t"," 我爱"]

# 移除列表中的空字符串
list3 = [s.strip() for s in list3 if s.strip() != ""]
print(list3)

['hello', '我爱']


In [31]:
list4 = [1,1,2,3,4,4,5]
# 使用set()来把列表中重复的元素去掉，再使用sorted()来排序
# 这样可以确保元素按升序排列，方便后续的索引操作
# 例如：list4[0] 表示第一个元素，list4[1] 表示第二个元素，以此类推
print(sorted(set(list4)))


{1, 2, 3, 4, 5}


In [23]:
text = "Hello, world. This, is a test."

# 捕获组：(\s) 表示匹配任意空白字符，并且会将匹配到的空白字符作为分隔符返回，所以列表中会包含空字符串
res = re.split(r'(\s)',text)
print(res)

# 复杂的正则表达式：通过[]指定要匹配的字符集，|表示或操作
res2 = re.split(r'([,.]|\s)',text)
print(res2)


['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']
['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [24]:
list1 = [1,2]
list2 = [3,4]
list1.extend(list2)
print(list1)

[1, 2, 3, 4]


In [39]:
list1 = ['a','ArithmeticError']
print(" ".join(list1))
print("\n".join(list1))
print("b".join(list1))

a ArithmeticError
a
ArithmeticError
abArithmeticError


In [25]:
vocab = {"hello":1,"world":2}
# 有元素返回就是true，这适用于列表
if vocab:
    print('not empty')
else:
    print('empty')

not empty


接下来处理数据
- 加载我们要处理的原始文本
- [伊迪丝·华顿的《判决书》](https://en.wikisource.org/wiki/The_Verdict) 是一个公共领域的短篇小说

In [26]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    # 将所有文本作为一个字符串读出
    raw_text = f.read()
print(type(raw_text))
print("总字符数:", len(raw_text))
print(raw_text[:99])

<class 'str'>
总字符数: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


- 目标是将这段文本进行分词和嵌入，以便用于语言模型（LLM）
- 基于正则表达式进行分词，规则简单，但是效果一般

In [30]:
preprocessed = re.split(r'([,.?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])
print(len(preprocessed))

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']
4649


## 2.3 将词元转换为词元IDs

- 从这些词元中，我们现在可以构建一个词汇表，它包含了所有独特的词元。
- 先得到唯一词元
- 然后构建词-id的映射

In [35]:
all_words = sorted(list(set(preprocessed)))
vocab_size = len(all_words)
print(vocab_size)

1159


In [36]:
vocab = {token:integer for integer,token in enumerate(all_words)}

- 以下是这个词汇表中的前50个条目：

In [38]:
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 10:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)


## 2.4 添加特殊上下文词元

- 一些分词器使用特殊词元来帮助语言模型（LLM）获取额外的上下文信息。
- 一些特殊的词元是
  - `[BOS]` (beginning of sequence) 标志着文本的开始
  - `[EOS]` (end of sequence) 标记文本结束的位置（通常用于连接多个不相关的文本，如两篇不同的维基百科文章或两本不同的书等）
  - `[PAD]` (padding) 如果我们训练 LLM 的批量大于 1（我们可能会包含多个长度不同的文本；使用填充标记，我们会将较短的文本填充为最长的文本，这样所有文本的长度就相等了。）
  - `[UNK]` 代表未列入词汇表的单词，为了OOV问题，我们可以在词汇表中添加特殊词元，如`"<|unk|>"`，以表示未知词

- 请注意，GPT-2 不需要上述任何标记，而只使用 `<|endoftext|>`来降低复杂性
-  `<|endoftext|>` 类似于上文提到的`[EOS]`
-  GPT 也使用 `<|endoftext|>` 作为填充词元（因为在批量输入训练时我们通常会使用掩码，反正我们不会关注填充的词元，所以这些词元是什么并不重要）。
-  GPT-2 不使用 `<UNK>` 词元来处理词汇表之外的单词；相反，GPT-2 使用字节对编码（BPE）分词器，它将单词分解为子词单元，我们将在后面的部分讨论这一点。
- 将所有这些整合到一个分词器类（tokenizer class）中
- 我们可以使用分词器将本文编码成整数
- 这些整数随后可以作为语言模型（LLM）的输入进行嵌入。

In [44]:
preprocessed = re.split(r'([,.?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]

all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}
print(len(vocab.items()))

for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

1161
('younger', 1156)
('your', 1157)
('yourself', 1158)
('<|endoftext|>', 1159)
('<|unk|>', 1160)


- 我们还需要相应调整分词器，以便它知道何时以及如何使用新的 `<unk>` 词元

In [45]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [item if item in self.str_to_int 
                        else "<|unk|>" for item in preprocessed]

        # ids = [self.str_to_int.get(s, '<|unk|>') for s in preprocessed]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # 替换指定标点符号前的空格
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [46]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [47]:
tokenizer.encode(text)

[1160,
 5,
 362,
 1155,
 642,
 1000,
 10,
 1159,
 57,
 1013,
 981,
 1009,
 738,
 1013,
 1160,
 7]

In [48]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.'

## 2.5 BytePair encoding（字节对编码）

- GPT-2 使用字节对编码（BPE）作为分词器
- 它允许模型将不在其预定义词汇表中的单词分解为更小的子单词单元甚至单个字符，从而使其能够处理词汇表之外的单词
- 例如，如果 GPT-2 的词汇表中没有 "unfamiliarword "这个词，它可能会根据训练有素的 BPE 合并结果，将其标记为["unfam"、"iliar"、"word"]或其他子词
- 原始 BPE 分词器可在此处找到：[https://github.com/openai/gpt-2/blob/master/src/encoder.py](https://github.com/openai/gpt-2/blob/master/src/encoder.py)
- 在本章中，我们使用 OpenAI 的开源 [tiktoken](https://github.com/openai/tiktoken) 库中的 BPE 标记符号生成器，该库用 Rust 实现了其核心算法，以提高计算性能
- 我在 [./bytepair_encoder](./bytepair_encoder)中创建了一个笔记本，并列比较了这两种实现方法（tiktoken 在样本文本上的速度大约快 5 倍）。

In [49]:
tokenizer = tiktoken.get_encoding("gpt2")

In [50]:
text = "Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace."

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

print(tokenizer.decode([50256]))

strings = tokenizer.decode(integers)

print(strings)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]
<|endoftext|>
Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.


- 未知单词实验：

In [51]:
integers = tokenizer.encode("Akwirw ier")
print(integers)

[33901, 86, 343, 86, 220, 959]


In [35]:
for i in integers:
    print(f"{i} -> {tokenizer.decode([i])}")

33901 -> Ak
86 -> w
343 -> ir
86 -> w
220 ->  
959 -> ier


In [43]:
strings = tokenizer.decode(integers)
print(strings)

Akwirw ier


## 2.6 使用滑动窗口进行数据采样

In [10]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


- 对于每个文本块，我们需要输入和目标
- 由于我们希望模型预测下一个单词，因此目标是向右移动一个位置的输入值

In [11]:
enc_sample = enc_text[50:]

context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


- 逐一预测的结果如下：

In [12]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "---->", desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [13]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


- 我们将在下一章介绍注意力机制后再处理下一个单词的预测
- 现在，我们实现了一个简单的数据加载器，它可以遍历输入数据集，并返回输入和目标值相移一的结果

- 安装并导入 PyTorch（安装提示见附录 A）

In [14]:
import torch
print("PyTorch version:", torch.__version__)

PyTorch version: 2.14.1+cpu


- 创建数据集和数据加载器，从输入的文本数据集中提取数据块

In [52]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.tokenizer = tokenizer
        self.input_ids = []
        self.target_ids = []

        # 对全部文本进行分词
        token_ids = tokenizer.encode(txt, allowed_special={'<|endoftext|>'})

        # 使用滑动窗口将图书分块为最大长度的重叠序列
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [53]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True):

    # 分词器初始化
    tokenizer = tiktoken.get_encoding("gpt2")

    # 创建数据集
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # 创建加载器
    dataloader = DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last)

    return dataloader

- 让我们用批量大小为 1 的数据加载器来测试上下文大小为 4 的 LLM：

In [54]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [55]:
dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [48]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


- 我们也可以创造批量的输出
-  请注意，我们在这里增加了步长，以避免批次之间的重叠，因为更多的重叠可能会导致过拟合增加。

In [49]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


## 2.7 创建词元嵌入

 - 数据已经几乎准备好用于大型语言模型（LLM）了
- 但最后，让我们使用嵌入层将这些词元嵌入到连续的向量表示中
- 通常，这些嵌入层是大型语言模型本身的一部分，在模型训练期间会进行更新（训练）

 - 假设我们有以下三个输入示例，输入ID分别为5、1、3和2（在进行分词之后）：

In [50]:
input_ids = torch.tensor([5, 1, 3, 2])

 - 为了简化问题，假设我们有一个很小的词汇表，只包含6个单词，我们想要创建大小为3的嵌入向量。

In [51]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

 - 这将导致一个6行3列的权重矩阵：

In [52]:
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


 - 对于那些熟悉独热编码（one-hot encoding）的人来说，上述的嵌入层方法本质上只是一种更高效的实现方式，它相当于在全连接层中先进行独热编码，然后进行矩阵乘法，这在补充代码[./embedding_vs_matmul](./embedding_vs_matmul)中有描述。
- 因为嵌入层只是一种更高效的实现方式，它等同于独热编码和矩阵乘法的方法，所以它可以被视为一个可以通过反向传播进行优化的神经网络层。

 - 要将ID为3的标记转换为三维向量，我们执行以下操作：

In [53]:
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


 - 请注意，上述内容是`embedding_layer`权重矩阵的第四行。
- 要嵌入上述所有三个`input_ids`的值，我们执行以下操作：

In [54]:
print(embedding_layer(input_ids))

tensor([[-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010],
        [-0.4015,  0.9666, -1.1481],
        [ 1.2753, -0.2010, -0.1606]], grad_fn=<EmbeddingBackward0>)


## 2.8 单词位置编码

- BytePair编码器有一个词汇量大小为50,257的词典
- 假设我们想要将输入的词元编码成256维的向量表示：

In [55]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

 - 如果我们从数据加载器中采样数据，我们将每个批次中的词元嵌入到一个256维的向量中。
- 如果我们的批次大小为8，每个批次有4个token，这将导致一个8 x 4 x 256的张量。

In [56]:
max_length = 4
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=max_length, stride=5, shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [57]:
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 3619,   402,   271, 10899],
        [  257,  7026, 15632,   438],
        [  257,   922,  5891,  1576],
        [  568,   340,   373,   645],
        [ 5975,   284,   502,   284],
        [  326,    11,   287,   262],
        [  286,   465, 13476,    11]])

Inputs shape:
 torch.Size([8, 4])


In [58]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


 - GPT-2 使用绝对位置嵌入，所以我们只需创建另一个嵌入层：

In [61]:
block_size = max_length
pos_embedding_layer = torch.nn.Embedding(block_size, output_dim)

In [62]:
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


 - 为了创建在大型语言模型（LLM）中使用的输入嵌入，我们只需将词元嵌入和位置嵌入相加：

In [63]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])
